# Node-level displacement prediction

This notebook trains one graph neural network across many truss designs and predicts displacement components at every node.


In [ ]:
from pathlib import Path
import copy
import os
import sys

import numpy as np
import pandas as pd
import torch
from torch_geometric.loader import DataLoader

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
elif not (PROJECT_ROOT / "src").exists():
    raise RuntimeError("Run this notebook from trussgraph_merged or trussgraph_merged/notebooks.")

SRC_ROOT = PROJECT_ROOT / "src"
if str(SRC_ROOT) not in sys.path:
    sys.path.insert(0, str(SRC_ROOT))

from trussgraph.dataset import TrussDisplacementDataset
from trussgraph.model import TrussDisplacementGNN
from trussgraph.training import (
    EpochSubsetSampler,
    checkpoint_path_for_config,
    load_model_checkpoint,
    save_model_checkpoint,
    collect_displacement_predictions,
    evaluate_displacement_loss,
    grouped_split,
    regression_metrics,
    seed_everything,
    train_displacement_epoch,
)
from trussgraph.visualization import (
    plot_deformed_truss,
    plot_displacement_parity,
    plot_displacement_residuals,
    plot_graph_relative_l2,
    plot_max_displacement_distribution,
    plot_panel_counts,
    plot_panel_performance,
    plot_training_history,
)

SEED = 42
seed_everything(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Project root: {PROJECT_ROOT}")
print(f"Device: {DEVICE}")


## Configuration

The limits below control both preprocessing time and dataset size. A different selection configuration creates a different file under `data/processed`, so a small development cache does not overwrite a larger experiment.

In [ ]:
DATA_ROOT = PROJECT_ROOT / "data"
TYPOLOGIES = ["Pratt"]     # Selected missing folders are downloaded from Zenodo.
PANELS = [8]               # Example: [6, 8, 10]
PLANE = "xz"

MAX_SEEDS_PER_PANEL = 3
MAX_DESIGNS_PER_SEED = 5000
TRAIN_FRACTION = 0.34
VALIDATION_FRACTION = 0.34

TRAIN_GRAPHS_PER_EPOCH = None  # Example: 20_000
VAL_GRAPHS_PER_CHECK = None    # Example: 3_000

BATCH_SIZE = 256
NUM_WORKERS = 0
HIDDEN_DIM = 64
NUM_LAYERS = 4
DROPOUT = 0.10
LEARNING_RATE = 1e-3
WEIGHT_DECAY = 1e-5
MAX_EPOCHS = 100
EARLY_STOPPING_PATIENCE = 15

CHECKPOINT_DIR = PROJECT_ROOT / "checkpoints"
CHECKPOINT_DIR.mkdir(exist_ok=True)


## PyG dataset

For every truss, the dataset creates a `torch_geometric.data.Data` object. Then they are collated into a single `torch\_geometric.data.InMemoryDataset` and stored as a processed version under `data/processed`.

### Node inputs

$$
\left[
\frac{x-x_c}{L_s},\;
\frac{z-z_c}{L_s},\;
\frac{F_x}{F_s},\;
\frac{F_z}{F_s},\;
\mathbb{1}_{\text{free}},\;
\frac{d_i}{d_{\max}}
\right]
$$

where \(L_s\) is the largest in-plane coordinate range and \(F_s\) is the sum of applied-load magnitudes.

### Edge inputs

Each physical member is represented in both directions. Its directed feature is:

$$
\left[\frac{\Delta x}{L_s},\frac{\Delta z}{L_s}\right].
$$

### Target normalization

With the nominal dataset assumption \(E=A=1\), linear-truss displacement scales with force and length. The target is therefore:

$$
\mathbf{u}_{\mathrm{norm}}=\frac{\mathbf{u}}{F_sL_s}.
$$

All test metrics are computed after converting predictions back to physical displacement units.

In [ ]:
dataset = TrussDisplacementDataset(
    root=DATA_ROOT,
    typologies=TYPOLOGIES,
    panels=PANELS,
    max_seeds_per_panel=MAX_SEEDS_PER_PANEL,
    max_designs_per_seed=MAX_DESIGNS_PER_SEED,
    sampling_seed=SEED,
    plane=PLANE,
)

metadata = dataset.metadata
print(f"Graphs: {len(dataset):,}")
print(f"Node features: {dataset.num_node_features}")
print(f"Edge features: {dataset.num_edge_features}")
print(f"Seed groups: {metadata['num_groups']:,}")
print(f"Processed cache: {dataset.processed_paths[0]}")
if metadata["num_skipped_files"]:
    print(f"Skipped malformed files: {metadata['num_skipped_files']}")


## Splits and loaders


In [ ]:
split = grouped_split(dataset, train_fraction=TRAIN_FRACTION, validation_fraction=VALIDATION_FRACTION, seed=SEED)
train_set = dataset[split.train]
validation_set = dataset[split.validation]
test_set = dataset[split.test]

TRAIN_EXCLUDE_KEYS = [
    "displacement", "displacement_scale", "member_index", "pos_raw",
    "typology_id", "panels", "seed", "group_id", "graph_id", "num_members",
]
loader_options = {"batch_size": BATCH_SIZE, "num_workers": NUM_WORKERS, "pin_memory": DEVICE.type == "cuda"}

train_sampler = None
if TRAIN_GRAPHS_PER_EPOCH is not None:
    train_sampler = EpochSubsetSampler(len(train_set), TRAIN_GRAPHS_PER_EPOCH, seed=SEED + 1)

validation_sampler = None
if VAL_GRAPHS_PER_CHECK is not None:
    validation_sampler = EpochSubsetSampler(len(validation_set), VAL_GRAPHS_PER_CHECK, seed=SEED + 2)

train_loader = DataLoader(
    train_set, shuffle=train_sampler is None, sampler=train_sampler,
    drop_last=train_sampler is not None, exclude_keys=TRAIN_EXCLUDE_KEYS, **loader_options
)
validation_loader = DataLoader(
    validation_set, shuffle=False, sampler=validation_sampler,
    exclude_keys=TRAIN_EXCLUDE_KEYS, **loader_options
)
test_loader = DataLoader(test_set, shuffle=False, **loader_options)

print(f"Train graphs:      {len(train_set):,}")
print(f"Validation graphs: {len(validation_set):,}")
print(f"Test graphs:       {len(test_set):,}")
print(f"Training batches per epoch: {len(train_loader):,}")


## Model

The model uses residual `GINEConv` layers. Unlike a plain GCN, GINE incorporates the directed geometric edge attributes during message passing. Four layers allow support and loading information to propagate across several neighboring members while remaining small enough for CPU experimentation.

The decoder predicts normalized displacement at every node. Predictions at fixed nodes are multiplied by zero, enforcing the known boundary condition exactly.


In [ ]:
model = TrussDisplacementGNN(
    node_dim=dataset.num_node_features,
    edge_dim=dataset.num_edge_features,
    hidden_dim=HIDDEN_DIM,
    num_layers=NUM_LAYERS,
    dropout=DROPOUT,
).to(DEVICE)
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="min", factor=0.5, patience=5)
print(model)
print(f"Trainable parameters: {sum(p.numel() for p in model.parameters()):,}")


## Training objective

The loss is Smooth L1 on normalized displacement. Errors are first averaged over the two displacement components and over the **free nodes of each graph**, then averaged across graphs in the batch.

This graph-balanced formulation prevents trusses with more nodes from automatically receiving more weight. Fixed nodes are excluded because their zero displacement is already enforced by the model.

In [ ]:
model_config = {
    "node_dim": dataset.num_node_features,
    "edge_dim": dataset.num_edge_features,
    "hidden_dim": HIDDEN_DIM,
    "num_layers": NUM_LAYERS,
    "dropout": DROPOUT,
}
model = TrussDisplacementGNN(**model_config).to(DEVICE)
checkpoint_path = checkpoint_path_for_config(
    CHECKPOINT_DIR,
    "truss_displacement_gnn",
    dataset.metadata,
    model_config,
    seed=SEED,
)
checkpoint = load_model_checkpoint(checkpoint_path, model, DEVICE)
print(model)
print(f"Trainable parameters: {sum(p.numel() for p in model.parameters()):,}")

if checkpoint is not None:
    history = checkpoint.get("history") or {"train_loss": [], "validation_loss": [], "learning_rate": []}
    best_validation_loss = float(checkpoint.get("best_validation_loss", "nan"))
    print("Loaded checkpoint:", checkpoint_path)
else:
    optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="min", factor=0.5, patience=5)
    history = {"train_loss": [], "validation_loss": [], "learning_rate": []}
    best_validation_loss = float("inf")
    best_state = None
    epochs_without_improvement = 0

    for epoch in range(1, MAX_EPOCHS + 1):
        train_loss = train_displacement_epoch(model, train_loader, optimizer, DEVICE)
        validation_loss = evaluate_displacement_loss(model, validation_loader, DEVICE)
        scheduler.step(validation_loss)
        history["train_loss"].append(train_loss)
        history["validation_loss"].append(validation_loss)
        history["learning_rate"].append(optimizer.param_groups[0]["lr"])

        if validation_loss < best_validation_loss:
            best_validation_loss = validation_loss
            best_state = copy.deepcopy(model.state_dict())
            epochs_without_improvement = 0
        else:
            epochs_without_improvement += 1

        if epoch == 1 or epoch % 5 == 0:
            print(f"Epoch {epoch:03d} | train {train_loss:.6f} | validation {validation_loss:.6f} | lr {optimizer.param_groups[0]['lr']:.2e}")
        if epochs_without_improvement >= EARLY_STOPPING_PATIENCE:
            print(f"Early stopping at epoch {epoch}")
            break

    if best_state is None:
        raise RuntimeError("Training did not produce a valid checkpoint")
    model.load_state_dict(best_state)
    save_model_checkpoint(
        checkpoint_path,
        model,
        model_config=model_config,
        dataset_metadata=dataset.metadata,
        best_validation_loss=best_validation_loss,
        seed=SEED,
        history=history,
    )
    print("Saved:", checkpoint_path)

if history.get("train_loss"):
    plot_training_history(history, log_y=True)
else:
    print("Training history is not available in this checkpoint.")


## Evaluation

Metrics are reported in the original physical displacement units and only over **free nodes**.

- **MAE — Mean Absolute Error:** average absolute prediction error. It is easy to interpret in the displacement unit and is less sensitive to a small number of large errors than RMSE.
- **RMSE — Root Mean Squared Error:** square root of the mean squared error. It penalizes large errors more strongly and is useful for identifying occasional poor predictions.
- **$R^2$ — Coefficient of determination:** compares the residual variance with the variance of the true targets. A value of 1 is perfect, 0 is equivalent to predicting the test-set mean, and a negative value is worse than that constant baseline.
- **Vector-magnitude metrics:** apply MAE, RMSE, and $R^2$ to $\|\mathbf u\|_2$, evaluating whether the model predicts total displacement magnitude independently of direction.

In [ ]:
results = collect_displacement_predictions(model, test_loader, DEVICE)
true_nodes = results["node_true"]
predicted_nodes = results["node_pred"]
true_magnitude = np.linalg.norm(true_nodes, axis=1)
predicted_magnitude = np.linalg.norm(predicted_nodes, axis=1)

metric_rows = []
for name, truth, prediction_values in [
    (f"U{PLANE[0]}", true_nodes[:, 0], predicted_nodes[:, 0]),
    (f"U{PLANE[1]}", true_nodes[:, 1], predicted_nodes[:, 1]),
    ("Displacement magnitude", true_magnitude, predicted_magnitude),
]:
    metric_rows.append({"Target": name, **regression_metrics(truth, prediction_values)})
metrics_table = pd.DataFrame(metric_rows).set_index("Target")
display(metrics_table.style.format({"MAE": "{:.6g}", "RMSE": "{:.6g}", "R2": "{:.4f}"}))

plot_displacement_parity(true_nodes, predicted_nodes, PLANE)
plot_displacement_residuals(true_nodes, predicted_nodes, PLANE)

## Deformed-shape example

The plot below overlays the true and predicted deformed truss. The deformation is automatically amplified for visibility; the numerical metrics above remain in physical units.


In [ ]:
example = dataset[split.test[0]].to(DEVICE)
model.eval()
with torch.no_grad():
    example_prediction = model(example) * example.displacement_scale
plot_deformed_truss(example.cpu(), example_prediction.cpu(), plane=PLANE)
